# Crop Recommendation: Training Notebook
Data Mining semester project. Sections are labeled by work day.
Put `Crop_recommendation.csv` next to this notebook (columns: N, P, K, temperature, humidity, ph, rainfall, label).
Everything is saved to `figures/` and `outputs/`.

## Setup (run first)

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")
SEED = 42
DATA_PATH = "Crop_recommendation.csv"
FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
TARGET = "label"
# Set True to inject ~2% missing values and a few extreme values so Preprocessing 1 has something to fix.
# If you use this, say so openly in the report (Dataset description, section 3.4).
ADD_ARTIFICIAL_NOISE = True

os.makedirs("figures", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

ModuleNotFoundError: No module named 'seaborn'

# WEDNESDAY: data, EDA, preprocessing

## 1. Load and explore (EDA)

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
display(df.head())
print(df.dtypes)
print(df.isna().sum())
display(df.describe().T)

In [ ]:
# Figure 1: class distribution
plt.figure(figsize=(10, 4))
df[TARGET].value_counts().plot(kind="bar", color="#2f6b4f")
plt.title("Class distribution"); plt.ylabel("Samples"); plt.tight_layout()
plt.savefig("figures/eda_class_distribution.png", dpi=150); plt.show()

# Figure 2: histograms
df[FEATURES].hist(bins=25, figsize=(12, 8), color="#2f6b4f")
plt.tight_layout(); plt.savefig("figures/eda_histograms.png", dpi=150); plt.show()

# Figure 3: boxplots
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes.ravel(), FEATURES):
    sns.boxplot(y=df[col], ax=ax, color="#9bbf8a"); ax.set_title(col)
axes.ravel()[-1].axis("off")
plt.tight_layout(); plt.savefig("figures/eda_boxplots.png", dpi=150); plt.show()

# Figure 4: correlation heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(df[FEATURES].corr(), annot=True, fmt=".2f", cmap="Greens")
plt.title("Correlation heatmap"); plt.tight_layout()
plt.savefig("figures/eda_correlation_heatmap.png", dpi=150); plt.show()

# Figure 5: average readings per crop
avg = df.groupby(TARGET)[FEATURES].mean()
avg[["N", "P", "K", "rainfall"]].plot(kind="bar", figsize=(14, 5))
plt.title("Average N, P, K and rainfall per crop"); plt.tight_layout()
plt.savefig("figures/avg_nutrients_per_crop.png", dpi=150); plt.show()

## 2. Optional: artificial noise
The Kaggle dataset is already clean. This cell injects a small, seeded amount of noise so the effect of Preprocessing 1 can be shown.
Report it honestly (percentages and seed).

In [ ]:
raw = df.copy()
if ADD_ARTIFICIAL_NOISE:
    rng = np.random.default_rng(SEED)
    n = len(raw)
    for col in FEATURES:
        idx = rng.choice(n, size=int(0.02 * n), replace=False)
        raw.loc[idx, col] = np.nan                      # about 2% missing per feature
    for col in ["rainfall", "K"]:
        idx = rng.choice(n, size=5, replace=False)
        raw.loc[idx, col] = raw[col].max() * 2          # 5 extreme values in two features
print("Missing per column:\n", raw.isna().sum())

## 3. Duplicates, then ONE stratified split
Rule used in the report: exact duplicates are checked before the split and removed from all three setups,
so every setup is tested on identical records.

In [ ]:
dups = raw.duplicated().sum()
print("Exact duplicate rows found:", dups)
raw = raw.drop_duplicates().reset_index(drop=True)

le = LabelEncoder()
y = pd.Series(le.fit_transform(raw[TARGET]), name="y")
X_raw = raw[FEATURES].copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, stratify=y, random_state=SEED)
print("Train:", X_train_raw.shape, " Test:", X_test_raw.shape)

## 4. The three setups
- **Setup A (raw):** no preprocessing. The models cannot accept missing values, so missing values are replaced by 0 only so they can run. This is the naive baseline.
- **Setup B (Preprocessing 1):** impute missing values with the training median, cap outliers with the IQR rule (limits computed on training data only; outliers are capped, not deleted, so rows stay identical).
- **Setup C (Preprocessing 1 + 2):** Setup B plus StandardScaler fitted on training data only.

In [ ]:
# Setup A
XA_train = X_train_raw.fillna(0)
XA_test = X_test_raw.fillna(0)

# Preprocessing 1: fit on training data only
medians = X_train_raw.median()
q1, q3 = X_train_raw.quantile(0.25), X_train_raw.quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

def preprocessing_1(X):
    X = X.fillna(medians)
    return X.clip(lower=lower, upper=upper, axis=1)

XB_train = preprocessing_1(X_train_raw)
XB_test = preprocessing_1(X_test_raw)

# Preprocessing 2: scaling, fitted on training data only
scaler = StandardScaler().fit(XB_train)
XC_train = pd.DataFrame(scaler.transform(XB_train), columns=FEATURES, index=XB_train.index)
XC_test = pd.DataFrame(scaler.transform(XB_test), columns=FEATURES, index=XB_test.index)

print("Missing values left in Setup B train:", int(XB_train.isna().sum().sum()))
print("Values capped by outlier rule (train):", int((XB_train != X_train_raw.fillna(medians)).sum().sum()))

# Figure: boxplots before/after Preprocessing 1 (training data)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=X_train_raw[["N", "P", "K", "ph"]], ax=axes[0]); axes[0].set_title("Before Preprocessing 1")
sns.boxplot(data=XB_train[["N", "P", "K", "ph"]], ax=axes[1]); axes[1].set_title("After Preprocessing 1")
plt.tight_layout(); plt.savefig("figures/boxplots_before_after_p1.png", dpi=150); plt.show()

# THURSDAY: models and evaluation

## 5. Compare six models on Setups A, B, C (same held-out records)

In [ ]:
def make_models():
    return {
        "Decision Tree": DecisionTreeClassifier(random_state=SEED),
        "Random Forest": RandomForestClassifier(n_estimators=200, random_state=SEED),
        "KNN": KNeighborsClassifier(),
        "Naive Bayes": GaussianNB(),
        "Logistic Regression": LogisticRegression(max_iter=3000),
        "SVM": SVC(probability=True, random_state=SEED),
    }

setups = {
    "A: Raw data": (XA_train, XA_test),
    "B: After Preprocessing 1": (XB_train, XB_test),
    "C: After Preprocessing 1 + 2": (XC_train, XC_test),
}

rows = []
for setup_name, (Xtr, Xte) in setups.items():
    for model_name, model in make_models().items():
        model.fit(Xtr, y_train)
        pred = model.predict(Xte)
        p, r, f1, _ = precision_recall_fscore_support(y_test, pred, average="weighted", zero_division=0)
        rows.append({"Setup": setup_name, "Model": model_name,
                     "Accuracy": accuracy_score(y_test, pred), "Precision": p, "Recall": r, "F1": f1})
results = pd.DataFrame(rows)
results.to_csv("outputs/model_comparison_all.csv", index=False)

acc_table = (results.pivot(index="Setup", columns="Model", values="Accuracy") * 100).round(2)
acc_table.to_csv("outputs/three_way_accuracy_table.csv")
display(acc_table)   # paste this into report section 8.1

## 6. Select the best model with cross-validation on TRAINING data only
The held-out test set is not used for choosing the model.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = {}
for name, model in make_models().items():
    s = cross_val_score(model, XC_train, y_train, cv=cv, scoring="accuracy")
    cv_scores[name] = (s.mean(), s.std())
cv_table = pd.DataFrame(cv_scores, index=["CV mean", "CV std"]).T.sort_values("CV mean", ascending=False)
display((cv_table * 100).round(2))
cv_table.to_csv("outputs/cv_scores_setup_c.csv")

best_name = cv_table.index[0]
print("Best model by 5-fold CV:", best_name)

# SATURDAY: tuning, feature importance, clustering

## 7. Tune the best model (GridSearchCV, training data only)

In [ ]:
param_grids = {
    "Random Forest": {"n_estimators": [100, 200, 400], "max_depth": [None, 10, 20], "min_samples_split": [2, 5]},
    "Decision Tree": {"max_depth": [None, 8, 12, 16], "min_samples_split": [2, 5, 10]},
    "KNN": {"n_neighbors": [3, 5, 7, 9], "weights": ["uniform", "distance"]},
    "SVM": {"C": [0.5, 1, 5, 10], "kernel": ["rbf", "linear"]},
    "Logistic Regression": {"C": [0.1, 1, 10, 100]},
    "Naive Bayes": {"var_smoothing": [1e-9, 1e-8, 1e-7]},
}
grid = GridSearchCV(make_models()[best_name], param_grids[best_name], cv=cv, scoring="accuracy", n_jobs=-1)
grid.fit(XC_train, y_train)
final_model = grid.best_estimator_
print("Best parameters:", grid.best_params_)
print("Best CV accuracy: %.4f" % grid.best_score_)

# Held-out evaluation, reported ONCE
pred = final_model.predict(XC_test)
print("Held-out accuracy: %.4f" % accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=le.classes_, zero_division=0))

cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion matrix: " + best_name); plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout(); plt.savefig("figures/confusion_matrix_best.png", dpi=150); plt.show()

## 8. Feature importance (permutation importance on held-out data)

In [ ]:
imp = permutation_importance(final_model, XC_test, y_test, n_repeats=10, random_state=SEED)
imp_df = pd.Series(imp.importances_mean, index=FEATURES).sort_values()
imp_df.plot(kind="barh", color="#2f6b4f", figsize=(7, 4))
plt.title("Feature importance (permutation, held-out)"); plt.tight_layout()
plt.savefig("figures/feature_importance.png", dpi=150); plt.show()
print(imp_df.sort_values(ascending=False))

## 9. Optional: K-Means on the scaled training data

In [ ]:
ks = range(2, 16)
inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(XC_train)
    inertia.append(km.inertia_); sil.append(silhouette_score(XC_train, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(ks), inertia, marker="o"); axes[0].set_title("Elbow method"); axes[0].set_xlabel("k")
axes[1].plot(list(ks), sil, marker="o", color="#2f6b4f"); axes[1].set_title("Silhouette score"); axes[1].set_xlabel("k")
plt.tight_layout(); plt.savefig("figures/elbow_silhouette.png", dpi=150); plt.show()

K = int(list(ks)[int(np.argmax(sil))])   # or choose from the elbow; justify in the report
print("Chosen k =", K)
km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(XC_train)
coords = PCA(n_components=2, random_state=SEED).fit_transform(XC_train)
plt.figure(figsize=(7, 5))
plt.scatter(coords[:, 0], coords[:, 1], c=km.labels_, cmap="tab20", s=12)
plt.title("K-Means clusters (PCA view)"); plt.tight_layout()
plt.savefig("figures/cluster_visualization.png", dpi=150); plt.show()

ct = pd.crosstab(km.labels_, le.inverse_transform(y_train.loc[XC_train.index]))
display(ct)    # which crops fall in which cluster

## 10. Choose the low-probability threshold from your own results

In [ ]:
# Out-of-fold probabilities on the training data
proba = cross_val_predict(grid.best_estimator_, XC_train, y_train, cv=cv, method="predict_proba")
top_p = proba.max(axis=1)
pred_cv = proba.argmax(axis=1)
correct = pred_cv == y_train.values
print("Top probability, correct predictions: median %.2f" % np.median(top_p[correct]))
if (~correct).any():
    print("Top probability, wrong predictions:   median %.2f, 90th percentile %.2f"
          % (np.median(top_p[~correct]), np.percentile(top_p[~correct], 90)))
for t in [0.4, 0.5, 0.6, 0.7]:
    flagged = top_p < t
    print(f"threshold {t:.0%}: flags {flagged.mean():.1%} of cases, "
          f"catches {(flagged & ~correct).sum()} of {(~correct).sum()} wrong predictions")
# Pick the threshold in app.py (LOW_PROB_THRESHOLD) from this output and report the value.

# SUNDAY: save everything the web app needs

In [ ]:
os.makedirs("artifacts", exist_ok=True)
joblib.dump(final_model, "artifacts/model.joblib", compress=3)
joblib.dump(scaler, "artifacts/scaler.joblib")
joblib.dump(le, "artifacts/label_encoder.joblib")

# Training range of each feature (cleaned training data, before scaling)
ranges = {"features": FEATURES,
          "ranges": {c: {"min": float(XB_train[c].min()), "max": float(XB_train[c].max())} for c in FEATURES}}
ranges["clip_lower"] = lower.to_dict()
ranges["clip_upper"] = upper.to_dict()
json.dump(ranges, open("artifacts/training_ranges.json", "w"), indent=2)

# Typical range of each feature for every crop (10th to 90th percentile of cleaned training data)
train_labels = le.inverse_transform(y_train.loc[XB_train.index])
profiles = {}
for crop in le.classes_:
    sub = XB_train[train_labels == crop]
    profiles[crop] = {c: {"low": float(sub[c].quantile(0.10)),
                          "median": float(sub[c].median()),
                          "high": float(sub[c].quantile(0.90))} for c in FEATURES}
json.dump(profiles, open("artifacts/crop_profiles.json", "w"), indent=2)

print(os.listdir("artifacts"))

## Final checklist
- Copy the numbers from `outputs/three_way_accuracy_table.csv` into report section 8.1.
- All charts are in `figures/` (names match the report's figure list).
- Copy the `artifacts/` folder next to `app.py`, then run `streamlit run app.py`.